## Load and prepare bulk-seq data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("imports done")

imports done


In [22]:
#load raw counts (DE-Seq2 uses raw counts)
counts = pd.read_csv(
    "data/raw/bulk/GSE193677_MSCCR_Biopsy_counts.txt.gz",
    sep=" ",
    index_col=0,
    quotechar='"'
)
print(counts.shape)
print(counts.iloc[:3, :3])


(56632, 2490)
                 MSCCR_reGRID_1_Biopsy_3  MSCCR_reGRID_1_Biopsy_2  \
ENSG00000000003                      704                     2855   
ENSG00000000005                        5                       18   
ENSG00000000419                      587                      563   

                 MSCCR_reGRID_1_Biopsy_1  
ENSG00000000003                     2088  
ENSG00000000005                        9  
ENSG00000000419                      685  


In [23]:
#load SOFT file
import gzip

samples = []
current = {}

with gzip.open("data/raw/bulk/GSE193677_family.soft.gz", "rt") as f:
    for line in f:
        line = line.strip()
        
        # new sample block starts
        if line.startswith("^SAMPLE"):
            if current:
                samples.append(current)
            current = {}
        
        # sample title = the MSCCR_reGRID name
        elif line.startswith("!Sample_title"):
            current["sample_title"] = line.split("= ", 1)[1]
        
        # characteristics lines - extract key and value
        elif line.startswith("!Sample_characteristics_ch1"):
            value = line.split("= ", 1)[1]  # everything after "= "
            if ": " in value:
                key, val = value.split(": ", 1)
                current[key.strip()] = val.strip()

# append last sample
if current:
    samples.append(current)

metadata = pd.DataFrame(samples)
print(metadata.shape)
print(metadata.columns.tolist())
print(metadata.head())

(2490, 22)
['sample_title', 'study_eligibility_age_at_endo', 'demographics_gender', 'regionre', 'diseasetypere', 'ibd_disease', 'typere', 'diseasebi', 'log2_fecalcalpro_mgperg', 'crp_jjmgl_log2', 'ibd_clinicianmeasure_inactive_active', 'ibd_endoseverity_4levels', 'ghas_sum7', 'nancyindex', 'ibdsescd_totalsescd', 'ibdmesuc_mayo_score', 'harveybradshawindex_hbi_score', 'colitisactivityindex_sccai', 'max_ghas_sum7', 'max_nancy', 'endoremiss', 'historemiss']
                                        sample_title  \
0  MSCCR_reGRID_1_Biopsy_1, UC participants,Rectu...   
1  MSCCR_reGRID_1_Biopsy_2, UC participants,LeftC...   
2  MSCCR_reGRID_1_Biopsy_3, UC participants,Ileum...   
3  MSCCR_reGRID_10_Biopsy_18, CD participants,Rig...   
4  MSCCR_reGRID_10_Biopsy_19, CD participants,Lef...   

  study_eligibility_age_at_endo demographics_gender    regionre diseasetypere  \
0                            44                Male      Rectum       UC.NonI   
1                            44           

In [24]:
#prepare metadata
metadata["sample_id"] = metadata["sample_title"].str.split(",").str[0]
print(metadata[["sample_id", "ibd_disease", "regionre", "typere"]].head())
print(metadata[["ibd_disease","regionre"]].astype('category'))
metadata.to_csv("data/processed/bulk/metadata_GSE193677.csv", index=False)
print("saved")


                   sample_id ibd_disease    regionre typere
0    MSCCR_reGRID_1_Biopsy_1          UC      Rectum   NonI
1    MSCCR_reGRID_1_Biopsy_2          UC   LeftColon   NonI
2    MSCCR_reGRID_1_Biopsy_3          UC       Ileum   NonI
3  MSCCR_reGRID_10_Biopsy_18          CD  RightColon   NonI
4  MSCCR_reGRID_10_Biopsy_19          CD   LeftColon   NonI
     ibd_disease    regionre
0             UC      Rectum
1             UC   LeftColon
2             UC       Ileum
3             CD  RightColon
4             CD   LeftColon
...          ...         ...
2485     Control      Rectum
2486     Control      Rectum
2487     Control       Cecum
2488     Control       Ileum
2489     Control      Rectum

[2490 rows x 2 columns]
saved


In [ ]:
#load metadata
metadata=pd.read_csv("data/processed/bulk/metadata_GSE193677.csv")

In [25]:
#extract Control+CD samples, save
meta = metadata[metadata["ibd_disease"].isin(["CD", "Control"])]
print(meta["ibd_disease"].value_counts())
print(meta["typere"].value_counts())
meta.to_csv("data/processed/bulk/meta_modeling.csv", index=False)
print(metadata["ibd_disease"].value_counts())

ibd_disease
CD         1157
Control     461
Name: count, dtype: int64
typere
NonI    1199
I        419
Name: count, dtype: int64
ibd_disease
CD         1157
UC          872
Control     461
Name: count, dtype: int64


In [26]:
#subset counts matrix to keep only CD and healthy samples
selected_samples=meta["sample_id"].tolist()
counts_selected=counts[selected_samples]
print("Samples in metadata:", len(meta["sample_id"]))
print("Samples in count matrix:", counts_selected.shape[1])
print("Match:", set(meta["sample_id"]) == set(counts_selected.columns.tolist()))


Samples in metadata: 1618
Samples in count matrix: 1618
Match: True


In [27]:
#rename ENSEMBL IDs genes to gene symbols
import mygene
mg = mygene.MyGeneInfo()

ensembl_ids = counts_selected.index.tolist() #create list of ENSEMBL IDs - row names

result = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene", #input
    fields="symbol", #output
    species="human",
    as_dataframe=True
)

print(result.shape)
print(result.head())


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


20 input query terms found dup hits:	[('ENSG00000175711', 2), ('ENSG00000188660', 2), ('ENSG00000215156', 2), ('ENSG00000226506', 2), ('E
7302 input query terms found no hit:	['ENSG00000002079', 'ENSG00000005955', 'ENSG00000006074', 'ENSG00000006075', 'ENSG00000006114', 'ENS


(56654, 4)
                   _id     _score  symbol notfound
query                                             
ENSG00000000003   7105  32.915176  TSPAN6      NaN
ENSG00000000005  64102  32.915142    TNMD      NaN
ENSG00000000419   8813  32.915010    DPM1      NaN
ENSG00000000457  57147  32.914740   SCYL3      NaN
ENSG00000000460  55732  32.914543   FIRRM      NaN


In [28]:
#clean output
gene_map=result[result["symbol"].notna()]
gene_map = gene_map[~gene_map.index.duplicated(keep="first")]
gene_map.head()

,_id,_score,symbol,notfound
query,,,,
ENSG00000000003,7105,32.915176,TSPAN6,NaN
ENSG00000000005,64102,32.915142,TNMD,NaN
ENSG00000000419,8813,32.915010,DPM1,NaN
ENSG00000000457,57147,32.914740,SCYL3,NaN
ENSG00000000460,55732,32.914543,FIRRM,NaN


In [32]:
#map IDs to counts matrix
counts_annotated=counts_selected.copy()
counts_annotated["symbol"]=gene_map["symbol"] #python automatically maps by index

# Drop rows with no symbol
counts_annotated = counts_annotated.dropna(subset=["symbol"])

# Set symbol as the new index
counts_annotated=counts_annotated.set_index("symbol")

#make sure there are no gene duplicates
counts_annotated = counts_annotated[~counts_annotated.index.duplicated(keep="first")]

print(counts_annotated.shape)
print(counts_annotated.index[:5].tolist())

counts_annotated.to_csv("data/processed/bulk/counts_mapped_ids.csv")
print("saved")

(40317, 1618)
['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'FIRRM']
saved


## Differential expression analysis

In [3]:
#load metadata and mapped counts
meta=pd.read_csv("data/processed/bulk/meta_modeling.csv")
counts_annotated = pd.read_csv("data/processed/bulk/counts_mapped_ids.csv")

In [33]:
#prepare to run DE analysis
# Make sure metadata is in same order as count matrix columns
meta_ordered = meta.set_index("sample_id").loc[counts_annotated.columns]
print(meta_ordered["ibd_disease"].value_counts())

#pip install pydeseq2

ibd_disease
CD         1157
Control     461
Name: count, dtype: int64


In [16]:
#clean environment
del metadata
del result
del counts
del counts_selected
del ensembl_ids
del gene_map

In [34]:
#run DESeq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# DESeq2 expects samples as rows, genes as columns — transpose
counts_for_deseq = counts_annotated.T.astype(int)

# create DESeq2 dataset. Reference is assigned here at random, proper stats and LFC are calculated in the next step
dds = DeseqDataSet(
    counts=counts_for_deseq,
    metadata=meta_ordered,
    design_factors="ibd_disease"
)

# Run DESeq2
dds.deseq2()

/var/folders/5x/6cxhjcxs5p79bg6pfvths8tr0000gn/T/ipykernel_79235/3643580500.py:9: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 2.82 seconds.

Fitting dispersions...
... done in 41.17 seconds.

Fitting dispersion trend curve...
... done in 5.95 seconds.

Fitting MAP dispersions...
... done in 67.84 seconds.

Fitting LFCs...
... done in 33.10 seconds.

Calculating cook's distance...
... done in 17.15 seconds.

Replacing 512 outlier genes.

Fitting dispersions...
... done in 4.34 seconds.

Fitting MAP dispersions...
... done in 1.84 seconds.

Fitting LFCs...
... done in 2.07 seconds.



In [35]:
#calculate statistics
from pydeseq2.ds import DeseqStats

stat_res = DeseqStats(dds, contrast=["ibd_disease", "CD", "Control"])
stat_res.summary()

results_df = stat_res.results_df
print(results_df.shape)
print(results_df.head())
results_df.to_csv("data/processed/bulk/DESeq_stats.csv")

Running Wald tests...
... done in 7.42 seconds.



Log2 fold change & Wald test p-value: ibd_disease CD vs Control
                 baseMean  log2FoldChange     lfcSE      stat        pvalue  \
symbol                                                                        
TSPAN6        2190.420460       -0.155046  0.038042 -4.075594  4.589711e-05   
TNMD            13.491963       -0.062171  0.061033 -1.018644  3.083721e-01   
DPM1           736.150422       -0.007602  0.019090 -0.398207  6.904778e-01   
SCYL3          532.349606       -0.080224  0.012856 -6.240066  4.373871e-10   
FIRRM          258.012159        0.038598  0.017676  2.183698  2.898442e-02   
...                   ...             ...       ...       ...           ...   
RNF225           0.488251        0.356528  0.155726  2.289461  2.205256e-02   
EGLN2          315.439114        0.020944  0.020384  1.027489  3.041904e-01   
ZNF628-DT        0.389551       -0.101155  0.163395 -0.619084  5.358610e-01   
LOC124904767     0.020439        0.033502  1.299854  0.025773  9.79

In [49]:
#filter significant DEGs
degs = results_df[results_df["padj"] < 0.05].copy()
print(degs_relaxed.shape)

(14511, 6)


## Find overlaps with myeloid signatures and build training matrix

In [ ]:
#import myeloid signatures
import json
with open("data/processed/scrna/myeloid_cd_gene_lists.json", "r") as f:
    gene_lists = json.load(f)

print(gene_lists.keys())


dict_keys(['M0 Macrophages', 'Neutrophil 1', 'Neutrophil 2', 'M2.2 Macrophages', 'Mixed Macrophages', 'Neutrophil 3', 'DCs', 'M1 Macrophages', 'IDA Macrophages', 'M2 Macrophages', 'M0_Ribhi Macrophages', 'Inflammatory Monocytes'])


In [ ]:
#find overlaps with myeloid signatures and pull scoring genes
scoring_genes = {}
for cell_type, genes in gene_lists.items():
    overlap = set(genes) & set(degs.index)
    scoring_genes[cell_type] = list(overlap)

for ct, genes in scoring_genes.items():
    print(ct, len(genes))

M0 Macrophages 71
Neutrophil 1 80
Neutrophil 2 66
M2.2 Macrophages 60
Mixed Macrophages 65
Neutrophil 3 85
DCs 56
M1 Macrophages 74
IDA Macrophages 58
M2 Macrophages 57
M0_Ribhi Macrophages 77
Inflammatory Monocytes 66


In [ ]:
#create scores list and turn into a single df
scores = {}
for ct, genes in scoring_genes.items():
    scores[ct]=counts_annotated.loc[genes].mean(axis=0)
feature_matrix = pd.DataFrame(scores)
print(feature_matrix.shape)
print(feature_matrix.head())

(1618, 12)
                             M0 Macrophages  Neutrophil 1  Neutrophil 2  \
MSCCR_reGRID_10_Biopsy_18      13962.169014     1377.0500   2363.439394   
MSCCR_reGRID_10_Biopsy_19      11282.380282     1199.3500   2118.136364   
MSCCR_reGRID_10_Biopsy_20      10587.084507     1139.1875   2068.545455   
MSCCR_reGRID_100_Biopsy_201    13727.704225     1197.4000   2124.712121   
MSCCR_reGRID_100_Biopsy_202    14482.563380     1535.9000   2566.075758   

                             M2.2 Macrophages  Mixed Macrophages  \
MSCCR_reGRID_10_Biopsy_18            12471.65        3831.369231   
MSCCR_reGRID_10_Biopsy_19            10335.50        4110.369231   
MSCCR_reGRID_10_Biopsy_20             9479.95        3235.384615   
MSCCR_reGRID_100_Biopsy_201          11810.90        3880.646154   
MSCCR_reGRID_100_Biopsy_202          10932.80        3918.276923   

                             Neutrophil 3          DCs  M1 Macrophages  \
MSCCR_reGRID_10_Biopsy_18     1922.152941  1951.142857 

In [63]:
#assign disease status labels
feature_matrix["label"] = meta_ordered["ibd_disease"].values
print(feature_matrix["label"].value_counts())
feature_matrix.to_csv("data/processed/bulk/feature_matrix.csv")


label
CD         1157
Control     461
Name: count, dtype: int64
